In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: terminal/native paired transfer

**User-adopted recipe (2026-10-15): historical ALLBIT, terminal/CFG-clean amplitude .10, conditional-clean .02 at CFG5, with the complete fixed budget below.** Run all directly executes this recipe in the user's GPU runtime. The local package passed bounded A2/A3/A5 review; real execution had not occurred at publication.

One original yellow-sailboat seed/prompt, fresh ZERO-control50-step baseline with CFG5. Historical JOINT BASE and S are not reused or rebuilt. Fresh terminal VAE loss gradient d=-Pgrad/||Pgrad||. TERMINAL=z_ZERO+.10d; NATIVE injects conditional-clean .02d at update48 (zero based) with complete original UniPC history and then reevaluates the real final update49. No cold start, posterior lift, scan, projection, extra optimization or full32 gate. Record actual clean/CFG/native displacement; old cap1 and radius1 are not equivalent.

Nominal cost:1generation preparation,102 ordinary DiT calls,52 native updates,2VAE loads,1gradient decode/VJP+2ordinary decodes.138 forward chunks and46 nominal checkpoint replays;0VAE encode/DiT VJP/DiT recompute/codec. ZERO gradient RGB is reused.3endpoints retain264windows/4224chips/165metrics, all32 payload and22state rows. Gradient-objective reader adds88windows/1408chips/55metrics using the same ZERO RGB, without an extra decode or independent endpoint.2quality pairs retain181frames/180transitions and actual peak neighborhoods. No new independent sample.

Transformer offloads to CPU around VAE work; VAE is released before the native tail. Temporary boundary spool uses the proven checkpoint path; prior real geometry required roughly64.53GiB temporary disk, an estimate, not a gate. Actual calls, chunks, spool and storage are recorded. Extra loads/work after interruption remain visible.

Every completed native step saves full history; gradient/RGB and fixed readers commit as durable units.60-second snapshots preserve last completed work. Same-Drive Run all skips complete units and repairs reports without models or rescoring saved reader/quality metrics; only missing images are redrawn. Failed/interrupted VJP forward artifacts are labeled forward-only. Counts separate completed, failed and completion-unknown interruptions. Saved effective task parameters remain in use on resume. Missing committed evidence is reported. Real cross-Colab recovery remains untested.

Outcomes: terminal improves/native loses -> reject this transport construction; both improve -> limited support; terminal finite step fails -> cannot blame tail. Missing/undefined stays unresolved. No generalization, robustness or multi-step joint-method PASS.

This notebook loads its ordinary companion ZIP from the existing dev/local-joint-state-payload-v1 branch. Validation at publication is CPU/fake/static; the user performs the real top-to-bottom Colab Run all. Return the whole task folder including failures. Publication does not merge main or imply scientific PASS.


In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable GitHub branch/tag/ref.
FIXED_CONFIG = {'schema': 'local-joint-paired-transfer-v1', 'selection_status': 'ADOPTED_BY_USER_20261015', 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'}, 'generation': {'height': 320, 'width': 512, 'frames': 181, 'steps': 50, 'guidance_scale': 5, 'max_sequence_length': 512, 'prompt': 'locked camera, a small yellow ceramic sailboat gently drifting across a shallow clear glass tank on a stone tabletop, steady soft daylight, no people, no cuts', 'negative_prompt': 'text, watermark, logo, camera motion, cuts, multiple objects, flicker', 'seed': 2026100701}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5}, 'source': {'source_id': 'yellow_sailboat_dev_s2026100701', 'development_only': True}, 'views': ['ZERO', 'TERMINAL', 'NATIVE'], 'objective': 'historical_allbit_v1', 'amplitude': 0.1, 'control_index': 48, 'device': 'cuda', 'transformer_dtype': 'bfloat16'}
EXPECTED_COST = {'generation_prepare': 1, 'vae_load': 2, 'gradient_decode': 1, 'decoder_vjp': 1, 'ordinary_decode': 2, 'decode': 3, 'gradient_forward_chunks': 46, 'ordinary_chunks': 92, 'total_forward_chunks': 138, 'nominal_replay_chunks': 46, 'baseline_dit': 100, 'tail_dit': 2, 'dit': 102, 'gradient_dit': 0, 'dit_vjp': 0, 'dit_recompute': 0, 'native': 52, 'scheduler': 52, 'encode': 0, 'codec': 0, 'independent_samples': 0, 'existing_source_count': 1, 'new_source_endpoints': 3, 'endpoint_windows': 264, 'endpoint_chips': 4224, 'endpoint_metrics': 165, 'gradient_windows': 88, 'gradient_chips': 1408, 'gradient_metrics': 55, 'gradient_observations_reused_as_ZERO': True, 'quality_pairs': 2}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Paired-Transfer')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / 'resumable-paired-transfer-v1'  # Same logical task on every Run all.
OUTPUT.mkdir(exist_ok=True)
ATTEMPT_OUTPUT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT_OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-Paired-Transfer-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(ATTEMPT_OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (ATTEMPT_OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_paired_transfer_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))
        actual = json.loads((RUN_OUTPUT / 'result.json').read_text())
        fixed_slots.update(run_status=actual.get('status'),
            views=actual.get('views', fixed_slots['views']), counts=actual.get('counts'),
            ordinary_chunks=actual.get('ordinary_chunks'))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    views={name: dict(status='NOT_RUN', expected_windows=88, expected_chips=1408, expected_metrics=55)
           for name in ('ZERO', 'TERMINAL', 'NATIVE')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
write_json(ATTEMPT_OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Fixed PAIRED-TRANSFER output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_paired_transfer_v1_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    # A repeated Run all may retain packages from an earlier extracted workspace.
    import importlib
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(ATTEMPT_OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
from runtime.wan.local_joint_paired_transfer_v1 import model_work_remaining
try:
    NEEDS_MODEL = model_work_remaining(RUN_OUTPUT)
    probe_path = ATTEMPT_OUTPUT / 'dependency_probe.json'
    probe_lines = [
        "import json, pathlib, sys, torch, numpy, PIL",
        "row = dict(python=sys.version, torch=str(torch.__version__), model_work_remaining=" + repr(NEEDS_MODEL) + ")",
    ]
    if NEEDS_MODEL:
        probe_lines += ["from diffusers import AutoencoderKLWan, WanPipeline, UniPCMultistepScheduler"]
        if str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
            probe_lines += ["assert torch.cuda.is_available(), 'CUDA is needed for the unfinished fixed model work'",
                            "row.update(cuda_available=True, device=torch.cuda.get_device_name(0))"]
    probe_lines += ["try:", "    pathlib.Path(" + repr(str(probe_path)) + ").write_text(json.dumps(row))",
                    "except Exception as exc:", "    print('Optional dependency note unavailable:', repr(exc), flush=True)"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        packages = ['torch', 'numpy', 'pillow']
        if NEEDS_MODEL:
            packages += ['diffusers', 'transformers', 'accelerate', 'safetensors',
                         'huggingface-hub', 'sentencepiece', 'ftfy']
        logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    try:
        environment_receipt = json.loads(probe_path.read_text())
        if not isinstance(environment_receipt, dict): raise ValueError('environment note must be an object')
    except Exception as exc:
        print('Optional dependency note unavailable:', repr(exc), flush=True)
        environment_receipt = dict(environment_note_error=repr(exc))
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode,
        source_and_versions_are_location_notes=True, checkpoint_interval_seconds=60)
    try:
        write_json(ATTEMPT_OUTPUT / 'environment_receipt.json', environment_receipt)
    except Exception as exc:
        print('Optional environment receipt unavailable:', repr(exc), flush=True)
    print('Remaining model work:', NEEDS_MODEL, '; task:', OUTPUT, flush=True)
except BaseException as exc:
    record_failure('ENVIRONMENT_SETUP', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_paired_transfer_v1_run',
               '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_PAIRED_TRANSFER', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_paired_transfer_v1 import finalize_interrupted
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.exists() else None
    write_json(ATTEMPT_OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] not in ('COMPLETE', 'COMPLETE_WITH_UNDEFINED_ARMS', 'COMPLETE_WITH_MISSING_EVIDENCE'):
        raise RuntimeError('PAIRED-TRANSFER engineering interruption; retain entire output: ' + str(OUTPUT))
    fixed_slots.update(status=result['status'], result='run/result.json',
        views=result['views'], counts=result['counts'], ordinary_chunks=result['ordinary_chunks'], units=result['units'], steps=result['steps'], quality=result['quality'])
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_PAIRED_TRANSFER', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
comparison = json.loads((RUN_OUTPUT / 'comparison.json').read_text())
print('Status:', result['status'], '; scientific PASS: False')
print('Selection:', result['selection_status'])
print('Nominal:', result['expected_cost'])
print('Cumulative:', result['counts'], 'current attempt:', result.get('current_attempt_counts'))
print('Replay:', result.get('physical_replay_chunks'), 'ordinary chunks:', result['ordinary_chunks'])
for name, row in comparison['views'].items():
    s = row.get('summary') or {}
    print(name, row['status'], 'positive=', s.get('positive_count'), 'trueG=', s.get('state_gap'),
          'allbit=', s.get('allbit_loss'), 'hard bits=', s.get('hard_bits'), 'reason=', row.get('reason'))
print('Coordinate receipts:', comparison['transport'])
print('Bounded outcome:', comparison['bounded_outcome'], comparison['changes'])
print('Quality:', comparison['quality'])
print('Failures retained:', result['failure_history'])
print(result['ceiling'])
print('Return whole directory:', OUTPUT)